In [0]:
# Setup
storage_account_name = "stkalshianalytics"
storage_account_key = dbutils.secrets.get("kalshi-scope", "adls-storage-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

print("Ready")

In [0]:
from pyspark.sql.functions import col, when, round as spark_round, current_timestamp

df_silver = spark.read.format("delta").load(f"{silver_path}/kalshi_markets")

df_gold = df_silver.select(
    "ticker",
    "title",
    "event_ticker",
    "status",
    "yes_ask",
    "yes_bid",
    "no_ask",
    "no_bid",
    "last_price",
    "liquidity",
    "volume_24h",
    "open_interest",
    "close_time",
    "is_provisional",
    "mve_collection_ticker",
    "no_sub_title",
    "yes_sub_title"
).withColumn(
    "implied_prob_yes",
    spark_round((col("yes_ask") + col("yes_bid")) / 2, 4)
).withColumn(
    "implied_prob_no",
    spark_round((col("no_ask") + col("no_bid")) / 2, 4)
).withColumn(
    "spread",
    spark_round(col("yes_ask") - col("yes_bid"), 4)
).withColumn(
    "is_single_market",
    col("mve_collection_ticker").isNull()
).withColumn(
    "has_liquidity",
    col("liquidity") > 0
).withColumn(
    "market_quality",
    when(col("liquidity") > 1000, "high")
    .when(col("liquidity") > 100, "medium")
    .when(col("liquidity") > 0, "low")
    .otherwise("none")
).withColumn(
    "gold_updated_at",
    current_timestamp()
)

print(f"Gold row count: {df_gold.count()}")
df_gold.filter(col("has_liquidity") == True).show(5, truncate=40)

In [0]:
# Write gold to Azure SQL
sql_server = "sql-edgar-analytics.database.windows.net"
sql_database = "edgar-analytics"
sql_user = "edgaradmin"
sql_password = dbutils.secrets.get("kalshi-scope", "sql-password")

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};user={sql_user};password={sql_password};encrypt=true;trustServerCertificate=false"

(df_gold.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "kalshi_gold.fact_markets")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("overwrite")
    .save()
)

print("Gold table written to Azure SQL successfully")